# Local Output Modifiers by Location

Computes per-location output modifiers for each good using the goods Ã— attributes matrix (goods_output_modifiers.xlsx). Not parsed from game filesâ€”derived from the Excel matrix and location attributes.

For each location: `local_{good}_output_modifier` = sum of matrix cells (topography + vegetation + climate + water). **Base output is 1.0.** Final output multiplier = `1.0 + local_{good}_output_modifier`. We also store `local_{good}_output_multiplier` = 1.0 + modifier for direct use.

In [29]:
import pandas as pd

from analysis.building_levels.building_analysis import CapacityAnalyzer, load_goods_output_modifiers

In [30]:
analyzer = CapacityAnalyzer()
df_locations = analyzer.location_data.get_merged_df()
matrix = load_goods_output_modifiers()

goods = list(matrix.index)

Loading live game data...


In [31]:
def compute_output_modifier(row, good, matrix_df):
    """Sum applicable matrix cells for this location and good."""
    topo = row.get("topography") or ""
    veg = row.get("vegetation") or ""
    climate = row.get("climate") or ""
    
    total = 0.0
    total += matrix_df.loc[good].get(topo, 0.0)
    total += matrix_df.loc[good].get(veg, 0.0)
    total += matrix_df.loc[good].get(climate, 0.0)
    
    if str(row.get("has_river", "no")).lower() == "yes":
        total += matrix_df.loc[good].get("has_river", 0.0)
    if str(row.get("is_adjacent_to_lake", "no")).lower() == "yes":
        total += matrix_df.loc[good].get("is_adjacent_to_lake", 0.0)
    if str(row.get("is_coastal", "no")).lower() == "yes":
        total += matrix_df.loc[good].get("is_coastal", 0.0)
    
    return total

for good in goods:
    col = f"local_{good}_output_modifier"
    df_locations[col] = df_locations.apply(
        lambda row, g=good: compute_output_modifier(row, g, matrix),
        axis=1,
    )
    # Base output = 1.0; add our sum to get the actual output multiplier
    df_locations[f"local_{good}_output_multiplier"] = 1.0 + df_locations[col]

## Output modifier stats (min, max, median, etc.)

In [32]:
mod_cols = [f"local_{g}_output_modifier" for g in goods if f"local_{g}_output_modifier" in df_locations.columns]
df_modifier_stats = df_locations[mod_cols].describe().T.drop(columns=["count"], errors="ignore")

# Color-coded for readability: gradient per column, high-contrast, larger font
df_modifier_stats.style.background_gradient(axis=0, cmap="RdYlGn").set_properties(
    **{"font-size": "13px", "font-weight": "500"}
).set_table_styles(
    [
        {"selector": "th", "props": [("font-size", "14px"), ("font-weight", "bold"), ("background", "#333"), ("color", "white")]},
        {"selector": "td", "props": [("padding", "8px")]},
    ]
)

,mean,std,min,25%,50%,75%,max
local_fruit_output_modifier,0.029633,0.333381,-1.000000,-0.200000,0.150000,0.300000,0.700000
local_fish_output_modifier,-0.097457,0.285484,-1.050000,-0.280000,0.050000,0.070000,0.420000
local_wool_output_modifier,0.021844,0.383188,-1.050000,-0.300000,0.050000,0.350000,0.750000
local_livestock_output_modifier,-0.053025,0.396324,-1.050000,-0.350000,-0.000000,0.250000,0.750000
local_millet_output_modifier,0.147368,0.261941,-0.700000,-0.050000,0.200000,0.350000,0.600000
local_wheat_output_modifier,-0.024236,0.398308,-1.050000,-0.300000,-0.050000,0.300000,0.800000
local_maize_output_modifier,0.041460,0.285159,-1.050000,-0.150000,0.050000,0.200000,0.700000
local_rice_output_modifier,0.055055,0.392094,-1.050000,-0.250000,0.150000,0.350000,0.750000
local_legumes_output_modifier,0.124447,0.289098,-1.000000,-0.050000,0.150000,0.350000,0.750000
local_potato_output_modifier,0.010268,0.363429,-1.050000,-0.350000,0.050000,0.350000,0.600000


### Same heatmap, RGO-filtered

For each good row, stats use **only** locations where `raw_material` is that good (the location’s natural RGO). Compare with the section above, which uses every location for every column.

In [33]:
def _normalize_rgo_id(val):
    if pd.isna(val) or val == "no_rgo":
        return None
    s = str(val).strip()
    if s.startswith("goods:"):
        return s[len("goods:") :]
    return s


_rgo_good = df_locations["raw_material"].map(_normalize_rgo_id)

rows_rgo = []
for col in mod_cols:
    # col is "local_{good}_output_modifier"
    good_id = col.removeprefix("local_").removesuffix("_output_modifier")
    sub = df_locations.loc[_rgo_good == good_id, col]
    rows_rgo.append(sub.describe())

df_modifier_stats_rgo = pd.DataFrame(rows_rgo, index=mod_cols)
df_modifier_stats_rgo = df_modifier_stats_rgo.drop(columns=["count"], errors="ignore")

print(
    "Locations per matrix good (RGO match):",
    {g: int((_rgo_good == g).sum()) for g in goods},
)

df_modifier_stats_rgo.style.background_gradient(axis=0, cmap="RdYlGn").set_properties(
    **{"font-size": "13px", "font-weight": "500"}
).set_table_styles(
    [
        {"selector": "th", "props": [("font-size", "14px"), ("font-weight", "bold"), ("background", "#333"), ("color", "white")]},
        {"selector": "td", "props": [("padding", "8px")]},
    ]
)

Locations per matrix good (RGO match): {'fruit': 823, 'fish': 1606, 'wool': 725, 'livestock': 2062, 'millet': 763, 'wheat': 953, 'maize': 275, 'rice': 538, 'legumes': 844, 'potato': 32, 'olives': 94, 'leather': 0, 'wild_game': 2005, 'fur': 912}


,mean,std,min,25%,50%,75%,max
local_fruit_output_modifier,0.033050,0.398913,-0.900000,-0.400000,0.250000,0.350000,0.650000
local_fish_output_modifier,0.009359,0.240086,-1.050000,-0.050000,0.070000,0.120000,0.420000
local_wool_output_modifier,0.257172,0.326960,-0.900000,0.050000,0.400000,0.500000,0.700000
local_livestock_output_modifier,0.123113,0.379268,-1.050000,-0.100000,0.150000,0.430000,0.750000
local_millet_output_modifier,0.270131,0.227812,-0.700000,0.150000,0.350000,0.450000,0.550000
local_wheat_output_modifier,0.296327,0.409835,-1.000000,0.000000,0.450000,0.600000,0.800000
local_maize_output_modifier,0.168545,0.267357,-0.500000,0.050000,0.100000,0.450000,0.600000
local_rice_output_modifier,0.356691,0.319759,-0.900000,0.200000,0.500000,0.600000,0.750000
local_legumes_output_modifier,0.272998,0.242340,-0.600000,0.150000,0.350000,0.430000,0.700000
local_potato_output_modifier,0.201563,0.300130,-1.000000,0.000000,0.350000,0.400000,0.450000


## Sample locations with output multipliers (1.0 + modifier)

## Location Power Potential (Capacity × Best Good Multiplier)

For each building: capacity × best output multiplier among its goods (cap at 0 if best < 0). Two variants: **with pop/dev** (full capacity) and **without pop/dev** (base + env + rank only). Sum across buildings = total_rural_potential and total_rural_potential_fixed.

In [34]:
# Building → goods mapping (from pp_building_capacity_values)
BUILDING_GOODS = {
    "fruit_orchard": ["fruit"],
    "fishing_village": ["fish"],
    "sheep_farms": ["wool"],
    "farming_village": ["livestock", "millet", "wheat", "maize", "rice", "legumes", "potato", "olives"],
    "forest_village": ["leather", "wild_game", "fur"],
}
# Display name → building_id for analyzer output
BUILDING_NAME_TO_ID = {
    "Fruit Orchard": "fruit_orchard",
    "Fishing Village": "fishing_village",
    "Sheep Farm": "sheep_farms",
    "Farming Village": "farming_village",
    "Forest Village": "forest_village",
}

# Get capacity per location per building
df_cap = analyzer.calculate_capacities_for_locations(df_locations)
# Full capacity (includes pop + dev bonus)
df_cap_pivot = df_cap.pivot_table(index="Location", columns="Building", values="Total Bonus", aggfunc="first")
# Fixed capacity (base + env + rank only, no pop/dev). Zero when env < 1 (not buildable).
df_cap["Fixed Bonus"] = df_cap.apply(lambda r: r["Base"] + r["Environmental Bonus"] + r["Rank Bonus"] if r["Environmental Bonus"] >= 1 else 0.0, axis=1)
df_cap_fixed_pivot = df_cap.pivot_table(index="Location", columns="Building", values="Fixed Bonus", aggfunc="first")

# For each building: best_mult = max(0, max(local_{good}_output_multiplier))
# Then potential = capacity × best_mult (both full and fixed)
for b_name, building_id in BUILDING_NAME_TO_ID.items():
    goods_list = BUILDING_GOODS[building_id]
    mult_cols = [f"local_{g}_output_multiplier" for g in goods_list if f"local_{g}_output_multiplier" in df_locations.columns]
    if not mult_cols:
        continue
    best_mult = df_locations[mult_cols].max(axis=1).clip(lower=0.0)
    # Full potential (with pop + dev)
    cap_series = (
        df_locations["location"].map(df_cap_pivot[b_name].to_dict()).fillna(0.0)
        if b_name in df_cap_pivot.columns
        else pd.Series(0.0, index=df_locations.index)
    )
    df_locations[f"{building_id}_potential"] = cap_series * best_mult
    # Fixed potential (without pop + dev)
    cap_fixed_series = (
        df_locations["location"].map(df_cap_fixed_pivot[b_name].to_dict()).fillna(0.0)
        if b_name in df_cap_fixed_pivot.columns
        else pd.Series(0.0, index=df_locations.index)
    )
    df_locations[f"{building_id}_potential_fixed"] = cap_fixed_series * best_mult

# Total potential: both full (with pop/dev) and fixed (without)
potential_cols = [f"{bid}_potential" for bid in BUILDING_GOODS if f"{bid}_potential" in df_locations.columns]
potential_fixed_cols = [f"{bid}_potential_fixed" for bid in BUILDING_GOODS if f"{bid}_potential_fixed" in df_locations.columns]
df_locations["total_rural_potential"] = df_locations[potential_cols].sum(axis=1)
df_locations["total_rural_potential_fixed"] = df_locations[potential_fixed_cols].sum(axis=1)

### Sample and top locations by total_rural_potential

In [35]:
potential_cols = [f"{bid}_potential" for bid in BUILDING_GOODS if f"{bid}_potential" in df_locations.columns]
potential_fixed_cols = [f"{bid}_potential_fixed" for bid in BUILDING_GOODS if f"{bid}_potential_fixed" in df_locations.columns]
base_cols = ["location", "region", "topography", "vegetation", "climate"]
available = [c for c in base_cols if c in df_locations.columns]
# With pop + dev (full capacity)
cols_full = available + potential_cols + ["total_rural_potential"]
cols_full = [c for c in cols_full if c in df_locations.columns]
print("With pop + dev:")
display(df_locations[cols_full].head(20))
# Without pop + dev (fixed capacity)
cols_fixed = available + potential_fixed_cols + ["total_rural_potential_fixed"]
cols_fixed = [c for c in cols_fixed if c in df_locations.columns]
print("Without pop + dev (fixed):")
display(df_locations[cols_fixed].head(20))

With pop + dev:


,location,region,topography,vegetation,climate,fruit_orchard_potential,fishing_village_potential,sheep_farms_potential,farming_village_potential,forest_village_potential,total_rural_potential
0,stockholm,scandinavian_region,flatland,grasslands,continental,6.721,9.0740,15.5890,17.7975,5.9455,55.1270
1,norrtalje,scandinavian_region,flatland,forest,continental,0.000,9.9082,8.5840,11.0880,11.0880,40.6682
2,enkoping,scandinavian_region,flatland,grasslands,continental,7.300,0.0000,16.2360,23.5280,6.7160,53.7800
3,uppsala,scandinavian_region,flatland,farmland,continental,8.802,0.0000,13.2060,27.1600,6.5200,55.6880
4,kastelholm,scandinavian_region,flatland,grasslands,continental,7.800,19.0450,17.0000,19.2500,6.9000,69.9950
5,tierp,scandinavian_region,flatland,woods,continental,0.000,9.8070,9.1800,12.5860,9.2560,40.8290
6,heby,scandinavian_region,flatland,grasslands,continental,7.125,0.0000,16.0050,18.1900,11.1550,52.4750
7,nykoping,scandinavian_region,flatland,grasslands,continental,8.021,19.0840,17.2890,19.5475,7.0955,71.0370
8,kolmarden,scandinavian_region,flatland,forest,continental,0.000,0.0000,8.4535,10.9620,10.9620,30.3775
9,strangnas,scandinavian_region,flatland,woods,continental,0.000,0.0000,9.1950,12.6015,9.2690,31.0655


Without pop + dev (fixed):


,location,region,topography,vegetation,climate,fruit_orchard_potential_fixed,fishing_village_potential_fixed,sheep_farms_potential_fixed,farming_village_potential_fixed,forest_village_potential_fixed,total_rural_potential_fixed
0,stockholm,scandinavian_region,flatland,grasslands,continental,-1.30,6.50,5.10,7.0,-1.15,16.15
1,norrtalje,scandinavian_region,flatland,forest,continental,0.00,8.56,2.90,5.6,5.60,22.66
2,enkoping,scandinavian_region,flatland,grasslands,continental,1.25,0.00,8.25,15.3,1.15,25.95
3,uppsala,scandinavian_region,flatland,farmland,continental,1.35,0.00,4.65,17.5,1.00,24.50
4,kastelholm,scandinavian_region,flatland,grasslands,continental,1.30,16.90,8.50,10.5,1.15,38.35
5,tierp,scandinavian_region,flatland,woods,continental,0.00,8.40,3.00,6.2,3.90,21.50
6,heby,scandinavian_region,flatland,grasslands,continental,1.25,0.00,8.25,10.2,5.75,25.45
7,nykoping,scandinavian_region,flatland,grasslands,continental,1.30,16.90,8.50,10.5,1.15,38.35
8,kolmarden,scandinavian_region,flatland,forest,continental,0.00,0.00,2.90,5.6,5.60,14.10
9,strangnas,scandinavian_region,flatland,woods,continental,0.00,0.00,3.00,6.2,3.90,13.10


In [36]:
print("Top 30 by total_rural_potential (with pop/dev):")
display(df_locations.nlargest(30, "total_rural_potential")[cols_full])
print("Top 30 by total_rural_potential_fixed (without pop/dev):")
display(df_locations.nlargest(30, "total_rural_potential_fixed")[cols_fixed])

Top 30 by total_rural_potential (with pop/dev):


,location,region,topography,vegetation,climate,fruit_orchard_potential,fishing_village_potential,sheep_farms_potential,farming_village_potential,forest_village_potential,total_rural_potential
8771,jiaxing,east_china_region,flatland,farmland,subtropical,51.0600,9.4185,44.6040,64.6680,0.0000,169.7505
8773,haiyan,east_china_region,flatland,farmland,subtropical,41.6250,7.6755,36.1125,53.9750,0.0000,139.3880
8619,changshu,east_china_region,flatland,farmland,subtropical,40.3500,7.9170,34.9650,52.5300,0.0000,135.7620
8607,ningguo,east_china_region,hills,woods,subtropical,32.7150,0.0000,28.0935,33.8055,35.9745,130.5885
8599,liyang,east_china_region,flatland,farmland,subtropical,36.2250,6.9195,31.2525,52.9550,0.0000,127.3520
8765,linan,east_china_region,hills,woods,subtropical,32.8800,0.0000,28.2420,33.9760,30.3340,125.4320
8768,anji,east_china_region,hills,woods,subtropical,32.7150,0.0000,28.0935,33.8055,30.1745,124.7885
8772,chongde,east_china_region,flatland,farmland,subtropical,36.7500,6.9930,31.7250,48.4500,0.0000,123.9180
8631,jurong,east_china_region,flatland,farmland,subtropical,34.6800,6.6990,29.8620,51.2040,0.0000,122.4450
10471,lingshanwei,north_china_region,flatland,farmland,continental,21.5740,12.0250,27.8560,43.9380,15.4100,120.8030


Top 30 by total_rural_potential_fixed (without pop/dev):


,location,region,topography,vegetation,climate,fruit_orchard_potential_fixed,fishing_village_potential_fixed,sheep_farms_potential_fixed,farming_village_potential_fixed,forest_village_potential_fixed,total_rural_potential_fixed
5678,split,balkan_region,hills,grasslands,mediterranean,9.60,16.20,11.55,10.38,1.28,49.01
7139,baniyas,crescent_region,hills,grasslands,mediterranean,9.60,16.20,11.55,10.38,1.28,49.01
3084,ajaccio,italy_region,hills,grasslands,mediterranean,9.60,5.40,16.50,15.57,1.28,48.35
2397,apt,france_region,hills,grasslands,mediterranean,18.60,0.00,11.20,14.85,1.28,45.93
2406,brignoles,france_region,hills,grasslands,mediterranean,18.60,0.00,11.20,14.85,1.28,45.93
5955,arta,balkan_region,hills,grasslands,mediterranean,18.60,0.00,11.20,14.85,1.28,45.93
17746,al_bayda,egypt_region,hills,grasslands,mediterranean,18.60,0.00,11.20,14.85,1.28,45.93
17970,thagia,maghreb_region,hills,grasslands,mediterranean,18.60,0.00,11.20,14.85,1.28,45.93
18346,paarl,southern_africa_region,hills,grasslands,mediterranean,18.60,0.00,11.20,14.85,1.28,45.93
5947,corfu,balkan_region,hills,forest,mediterranean,17.05,5.48,5.80,11.41,5.92,45.66


In [37]:
mult_cols = [f"local_{g}_output_multiplier" for g in goods]
display_cols = ["location", "region", "topography", "vegetation", "climate", "is_coastal", "has_river", "is_adjacent_to_lake"] + mult_cols
available = [c for c in display_cols if c in df_locations.columns]
df_locations[available].head(20)

,location,region,topography,vegetation,climate,is_coastal,has_river,is_adjacent_to_lake,local_fruit_output_multiplier,local_fish_output_multiplier,...,local_millet_output_multiplier,local_wheat_output_multiplier,local_maize_output_multiplier,local_rice_output_multiplier,local_legumes_output_multiplier,local_potato_output_multiplier,local_olives_output_multiplier,local_leather_output_multiplier,local_wild_game_output_multiplier,local_fur_output_multiplier
0,stockholm,scandinavian_region,flatland,grasslands,continental,yes,no,no,1.30,1.30,...,1.50,1.75,1.15,1.45,1.50,1.50,1.08,1.0,1.10,1.15
1,norrtalje,scandinavian_region,flatland,forest,continental,no,no,no,1.20,1.07,...,1.30,1.40,1.00,1.35,1.35,1.30,0.90,1.0,1.30,1.40
2,enkoping,scandinavian_region,flatland,grasslands,continental,no,no,no,1.25,1.05,...,1.45,1.70,1.10,1.40,1.45,1.45,1.00,1.0,1.10,1.15
3,uppsala,scandinavian_region,flatland,farmland,continental,no,no,no,1.35,1.05,...,1.40,1.75,1.15,1.50,1.55,1.50,1.05,1.0,0.75,0.95
4,kastelholm,scandinavian_region,flatland,grasslands,continental,yes,no,no,1.30,1.30,...,1.50,1.75,1.15,1.45,1.50,1.50,1.08,1.0,1.10,1.15
5,tierp,scandinavian_region,flatland,woods,continental,no,no,no,1.25,1.05,...,1.35,1.55,1.05,1.35,1.38,1.35,0.93,1.0,1.30,1.30
6,heby,scandinavian_region,flatland,grasslands,continental,no,no,no,1.25,1.05,...,1.45,1.70,1.10,1.40,1.45,1.45,1.00,1.0,1.10,1.15
7,nykoping,scandinavian_region,flatland,grasslands,continental,yes,no,no,1.30,1.30,...,1.50,1.75,1.15,1.45,1.50,1.50,1.08,1.0,1.10,1.15
8,kolmarden,scandinavian_region,flatland,forest,continental,no,no,no,1.20,1.07,...,1.30,1.40,1.00,1.35,1.35,1.30,0.90,1.0,1.30,1.40
9,strangnas,scandinavian_region,flatland,woods,continental,no,no,no,1.25,1.05,...,1.35,1.55,1.05,1.35,1.38,1.35,0.93,1.0,1.30,1.30


## Curiosities: high capacity, unusually low yield

Locations where a building has **high capacity** (top 25%) but **unusually low max yield** (bottom 25%)—the terrain gives lots of slots but poor output modifiers for that building's goods.

In [38]:
from IPython.display import display

CAPACITY_TOP_PCT = 50   # capacity percentile threshold (high = top 40%)
YIELD_BOTTOM_PCT = 50   # yield percentile threshold (low = bottom 40%)

rows = []
for b_name, building_id in BUILDING_NAME_TO_ID.items():
    if b_name not in df_cap_pivot.columns:
        continue
    cap_col = f"{building_id}_capacity"
    yield_col = f"{building_id}_potential"
    if yield_col not in df_locations.columns:
        continue

    # Capacity from pivot (Location -> value)
    cap_series = df_locations["location"].map(df_cap_pivot[b_name].to_dict()).fillna(0.0)
    yield_series = df_locations[yield_col]

    # Only buildable locations (capacity > 0)
    mask = cap_series > 0
    cap_valid = cap_series[mask]
    yield_valid = yield_series[mask]
    if cap_valid.empty:
        continue

    cap_pct = cap_valid.rank(pct=True) * 100
    yield_pct = yield_valid.rank(pct=True) * 100

    # Curiosity: high capacity rank, low yield rank
    curiosity_mask = (cap_pct >= CAPACITY_TOP_PCT) & (yield_pct <= YIELD_BOTTOM_PCT)
    for idx in cap_valid[curiosity_mask].index:
        loc = df_locations.loc[idx, "location"]
        rows.append({
            "location": loc,
            "building": b_name,
            "capacity": cap_series.loc[idx],
            "yield": yield_series.loc[idx],
            "capacity_pct": cap_pct.loc[idx],
            "yield_pct": yield_pct.loc[idx],
            "best_mult": yield_series.loc[idx] / cap_series.loc[idx] if cap_series.loc[idx] > 0 else 0,
            **{c: df_locations.loc[idx, c] for c in ["region", "topography", "vegetation", "climate"] if c in df_locations.columns},
        })

df_curiosities = pd.DataFrame(rows)
if df_curiosities.empty:
    # Fallback: show "near misses" — high capacity locations with lowest yield ratio
    rows_fallback = []
    for b_name, building_id in BUILDING_NAME_TO_ID.items():
        if b_name not in df_cap_pivot.columns:
            continue
        yield_col = f"{building_id}_potential"
        if yield_col not in df_locations.columns:
            continue
        cap_series = df_locations["location"].map(df_cap_pivot[b_name].to_dict()).fillna(0.0)
        yield_series = df_locations[yield_col]
        mask = cap_series > 0
        if mask.sum() < 2:
            continue
        cap_valid, yield_valid = cap_series[mask], yield_series[mask]
        # Yield/capacity ratio; rank ascending = lowest ratio first
        ratio = yield_valid / cap_valid
        low_ratio = ratio.nsmallest(5)
        for idx in low_ratio.index:
            rows_fallback.append({
                "location": df_locations.loc[idx, "location"],
                "building": b_name,
                "capacity": cap_series.loc[idx],
                "yield": yield_series.loc[idx],
                "best_mult": yield_series.loc[idx] / cap_series.loc[idx],
                **{c: df_locations.loc[idx, c] for c in ["region", "topography", "vegetation", "climate"] if c in df_locations.columns},
            })
    df_fallback = pd.DataFrame(rows_fallback)
    print("No curiosities with current thresholds. Showing locations with high capacity but lowest yield ratio per building:")
    display(df_fallback)
else:
    df_curiosities = df_curiosities.sort_values(["building", "capacity_pct"], ascending=[True, False]).reset_index(drop=True)
    display(df_curiosities)

,location,building,capacity,yield,capacity_pct,yield_pct,best_mult,region,topography,vegetation,climate
0,leshou,Farming Village,8.39,8.3900,66.987078,48.688150,1.00,north_china_region,wetlands,grasslands,cold_arid
1,macaya,Farming Village,8.00,8.4000,63.492751,49.854239,1.05,andes_region,plateau,grasslands,arctic
2,daya,Farming Village,7.96,8.3580,63.144107,48.629058,1.05,indonesia_region,mountains,jungle,tropical
3,udayagiri,Farming Village,7.84,7.4480,62.450756,44.890482,0.95,deccan_region,hills,jungle,arid
4,levanger,Farming Village,7.83,8.2215,62.389694,48.195714,1.05,scandinavian_region,hills,grasslands,arctic
...,...,...,...,...,...,...,...,...,...,...,...
2712,east_grand_erg_oriental,Sheep Farm,6.40,4.8000,50.165587,37.547396,0.75,egypt_region,flatland_wasteland,sparse,arid
2713,hamada_el_hamra,Sheep Farm,6.40,4.8000,50.165587,37.547396,0.75,maghreb_region,plateau_wasteland,sparse,arid
2714,tademait_plateau2,Sheep Farm,6.40,4.8000,50.165587,37.547396,0.75,maghreb_region,flatland_wasteland,sparse,arid
2715,yauri,Sheep Farm,6.40,7.0400,50.165587,48.934485,1.10,sahel_region,flatland,grasslands,tropical
